<a href="https://colab.research.google.com/github/Usermer/deep-learning-universe/blob/main/tracking_optimized_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Traffic Analysis & Safety Monitoring
## Multi-Object Vehicle Tracking — Version Optimisée avec Filtrage Mouvement

**Améliorations clés :**
- ✅ Filtrage MOG2 : seuls les véhicules **en mouvement** sont gardés (alignement avec UA-DETRAC GT)
- ✅ NMS post-YOLO : suppression des détections dupliquées
- ✅ Filtrage taille : suppression des micro-détections bruitées
- ✅ Alignement frame_id parfait (pas de décalage)
- ✅ MOTA / MOTP / IDF1 calculés correctement

---

### Pourquoi le filtrage mouvement est indispensable

UA-DETRAC n'annote **que les véhicules en mouvement**. Sans filtrage :
- YOLO détecte ~27 véhicules/frame (voitures garées incluses)
- GT contient ~13 véhicules/frame (en mouvement uniquement)
- → Ratio 2x de faux positifs → MOTA effondré

Avec MOG2 on élimine les véhicules statiques → FP divisés par ~2 → MOTA +15 à +25 pts.

## 1. Installation

In [1]:
!pip install ultralytics boxmot lapx motmetrics seaborn -q
!wget -q https://github.com/mikel-brostrom/boxmot/releases/download/v10.0.43/osnet_x0_25_msmt17.pt
print('Installation terminée')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 44.9 MB/s eta 0:00:00
Installation terminée


## 2. Imports

In [2]:
import os, time, warnings, logging, json, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import motmetrics as mm

from ultralytics import YOLO
from boxmot.trackers.bytetrack.bytetrack   import ByteTrack
from boxmot.trackers.strongsort.strongsort import StrongSort
from boxmot.trackers.deepocsort.deepocsort import DeepOcSort

warnings.filterwarnings('ignore')
logging.getLogger('boxmot').setLevel(logging.ERROR)
np.asfarray = lambda x, dtype=float: np.asarray(x, dtype=dtype)  # fix NumPy 2.0

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}  |  Device : {DEVICE}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cpu  |  Device : cpu


In [8]:
import kagglehub
path = kagglehub.dataset_download('bratjay/ua-detrac-orig')
print(f'Dataset : {path}')

Using Colab cache for faster access to the 'ua-detrac-orig' dataset.
Dataset : /kaggle/input/ua-detrac-orig


### Suppression du cache et retéléchargement du dataset

In [14]:
import shutil
import os

# The BASE variable points to the dataset's cache directory
cache_dir = '/root/.cache/kagglehub/datasets/bratjay/ua-detrac-orig/versions/2'

if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f'Cache directory removed: {cache_dir}')
else:
    print(f'Cache directory not found: {cache_dir}')


Cache directory not found: /root/.cache/kagglehub/datasets/bratjay/ua-detrac-orig/versions/2


In [21]:
import kagglehub
path = kagglehub.dataset_download('bratjay/ua-detrac-orig')
print(f'Dataset : {path}')


Using Colab cache for faster access to the 'ua-detrac-orig' dataset.
Dataset : /kaggle/input/ua-detrac-orig


In [23]:
# BASE = '/kaggle/input/ua-detrac-orig'
# print(os.path.exists(BASE))  # doit afficher True
# import os

# BASE = '/kaggle/input/ua-detrac-orig'
# for root, dirs, files in os.walk(BASE):
#     level = root.replace(BASE, '').count(os.sep)
#     if level > 3:
#         dirs.clear()
#         continue
#     indent = '  ' * level
#     print(f"{indent}{os.path.basename(root)}/")
#     for f in sorted(files)[:2]:
#         print(f"{indent}  {f}")

True
ua-detrac-orig/
  DETRAC-Test-Annotations-XML/
    DETRAC-Test-Annotations-XML/
      MVI_39031.xml
      MVI_39051.xml
  DETRAC-MOT-toolkit/
    DETRAC-MOT-toolkit/
      DETRAC_experiment.m
      README.md
      trackers/
      utils/
        checkOptions.m
        deleteFolder.m
      evaluation/
        AP_DET_EVAL.exe
        CLEAR_MOT.m
  DETRAC-Images/
    DETRAC-Images/
      MVI_39501/
        img00001.jpg
        img00002.jpg
      MVI_40171/
        img00001.jpg
        img00002.jpg
      MVI_40192/
        img00001.jpg
        img00002.jpg
      MVI_20011/
        img00001.jpg
        img00002.jpg
      MVI_40892/
        img00001.jpg
        img00002.jpg
      MVI_40772/
        img00001.jpg
        img00002.jpg
      MVI_20033/
        img00001.jpg
        img00002.jpg
      MVI_20063/
        img00001.jpg
        img00002.jpg
      MVI_40901/
        img00001.jpg
        img00002.jpg
      MVI_39931/
        img00001.jpg
        img00002.jpg
      MVI_40773/
       

In [24]:
BASE         = '/kaggle/input/ua-detrac-orig'
IMG_DIR      = f'{BASE}/DETRAC-Images/DETRAC-Images'
XML_DIR      = f'{BASE}/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML'
OUT_DIR      = Path('/content/detrac_yolo')
YOLO_CLASS   = 0

# Vérifier
import os
xmls = list(Path(XML_DIR).glob('*.xml'))
imgs = [d for d in os.listdir(IMG_DIR) if os.path.isdir(f'{IMG_DIR}/{d}')]
print(f"XMLs test  : {len(xmls)}")
print(f"Séquences images : {len(imgs)}")
print(f"Exemple XML : {xmls[0].name if xmls else 'AUCUN'}")

XMLs test  : 40
Séquences images : 100
Exemple XML : MVI_39051.xml


In [25]:
from pathlib import Path

XML_TRAIN = '/kaggle/input/ua-detrac-orig/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML'
XML_TEST  = '/kaggle/input/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML'

train_xmls = list(Path(XML_TRAIN).glob('*.xml'))
test_xmls  = list(Path(XML_TEST).glob('*.xml'))
print(f"XMLs train : {len(train_xmls)}")
print(f"XMLs test  : {len(test_xmls)}")
print(f"Total      : {len(train_xmls) + len(test_xmls)}")

XMLs train : 60
XMLs test  : 40
Total      : 100


In [26]:
import os, shutil, cv2
from pathlib import Path
import xml.etree.ElementTree as ET

BASE    = '/kaggle/input/ua-detrac-orig'
IMG_DIR = f'{BASE}/DETRAC-Images/DETRAC-Images'
OUT_DIR = Path('/content/detrac_yolo')

XML_TRAIN = f'{BASE}/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML'
XML_TEST  = f'{BASE}/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML'

for split in ['train', 'val']:
    (OUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# ── Parser XML ────────────────────────────────────────────────────────────────
def parse_detrac_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ignored = []
    ir = root.find('ignored_region')
    if ir is not None:
        for box in ir.iter('box'):
            ignored.append((float(box.attrib['left']), float(box.attrib['top']),
                            float(box.attrib['width']), float(box.attrib['height'])))
    frames = {}
    for frame in root.iter('frame'):
        fid = int(frame.attrib['num'])
        boxes = []
        for box in frame.iter('box'):
            boxes.append((float(box.attrib['left']), float(box.attrib['top']),
                          float(box.attrib['width']), float(box.attrib['height'])))
        if boxes:
            frames[fid] = boxes
    return frames, ignored

def in_ignored(bx, by, bw, bh, ignored):
    for (ix, iy, iw, ih) in ignored:
        inter_w = max(0, min(bx+bw, ix+iw) - max(bx, ix))
        inter_h = max(0, min(by+bh, iy+ih) - max(by, iy))
        if bw * bh > 0 and (inter_w * inter_h) / (bw * bh) > 0.5:
            return True
    return False

# ── Conversion ────────────────────────────────────────────────────────────────
def convert_sequence(xml_path, split, max_frames=250):
    seq_name    = xml_path.stem
    img_seq_dir = Path(IMG_DIR) / seq_name
    if not img_seq_dir.exists():
        return 0, "SKIP – images manquantes"
    frames_gt, ignored = parse_detrac_xml(xml_path)
    if not frames_gt:
        return 0, "SKIP – pas d'annotations"
    imgs = sorted(img_seq_dir.glob('img*.jpg'))
    if not imgs:
        return 0, "SKIP – pas d'images"
    count = 0
    for img_path in imgs[:max_frames]:
        fid = int(''.join(filter(str.isdigit, img_path.stem)))
        if fid not in frames_gt:
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        lines = []
        for (x, y, w, h) in frames_gt[fid]:
            if in_ignored(x, y, w, h, ignored):
                continue
            cx, cy = (x + w/2) / W, (y + h/2) / H
            nw, nh = w / W, h / H
            cx, cy, nw, nh = [max(0., min(1., v)) for v in [cx, cy, nw, nh]]
            if nw > 0.005 and nh > 0.005:
                lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if not lines:
            continue
        fname = f"{seq_name}_{fid:05d}"
        shutil.copy(img_path, OUT_DIR / split / 'images' / f"{fname}.jpg")
        (OUT_DIR / split / 'labels' / f"{fname}.txt").write_text('\n'.join(lines))
        count += 1
    return count, "OK"

# ── Combiner train + test XMLs, splitter 85/15 ───────────────────────────────
all_xmls = sorted(Path(XML_TRAIN).glob('*.xml')) + sorted(Path(XML_TEST).glob('*.xml'))
split_idx = int(len(all_xmls) * 0.85)
print(f"Total XMLs : {len(all_xmls)} → train={split_idx} | val={len(all_xmls)-split_idx}")

total_train = total_val = 0
skipped = []

for i, xml_path in enumerate(all_xmls):
    split = 'train' if i < split_idx else 'val'
    n, msg = convert_sequence(xml_path, split)
    if msg != "OK":
        skipped.append((xml_path.stem, msg))
    elif split == 'train': total_train += n
    else:                  total_val   += n
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(all_xmls)}] train={total_train} val={total_val}")

print(f"\n✅ Conversion terminée")
print(f"   Train : {total_train} | Val : {total_val}")
if skipped:
    print(f"   Ignorées : {len(skipped)} séquences")
    for s, m in skipped[:5]: print(f"   {s} → {m}")

Total XMLs : 100 → train=85 | val=15
  [10/100] train=2500 val=0
  [20/100] train=4867 val=0
  [30/100] train=7367 val=0
  [40/100] train=9853 val=0
  [50/100] train=12353 val=0
  [60/100] train=14853 val=0
  [70/100] train=17349 val=0
  [80/100] train=19849 val=0
  [90/100] train=21099 val=1250
  [100/100] train=21099 val=3750

✅ Conversion terminée
   Train : 21099 | Val : 3750


## 3. Configuration

In [ ]:
# ── Chemins ──────────────────────────────────────────────────────────────────
BASE         = '/root/.cache/kagglehub/datasets/bratjay/ua-detrac-orig/versions/2'
IMG_DIR      = f'{BASE}/DETRAC-Images/DETRAC-Images'
XML_DIR_TEST = f'{BASE}/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML'
YOLO_WEIGHTS = '/content/yolov8s.pt'
REID_WEIGHTS = Path('osnet_x0_25_msmt17.pt')

# ── Séquences ────────────────────────────────────────────────────────────────
TEST_SEQUENCES = ['MVI_39311']
SEQ_TEST       = TEST_SEQUENCES[0]

# ── Hyperparamètres détecteur ─────────────────────────────────────────────────
CONF_THRESHOLD  = 0.25          # seuil bas → plus de rappel, MOG2 filtre les FP
IOU_THRESHOLD   = 0.45
VEHICLE_CLASSES = [2, 5, 7]     # COCO : car=2, bus=5, truck=7

# ── Matching GT/Pred ──────────────────────────────────────────────────────────
IOU_MATCH_THRESHOLD = 0.50

# ── Filtrage mouvement (MOG2) ─────────────────────────────────────────────────
MOG2_HISTORY        = 100        # nb frames pour modéliser le fond
MOG2_VAR_THRESHOLD  = 25       # sensibilité au mouvement
MOG2_MOTION_RATIO   = 0.02   # % min de pixels en mouvement dans la box

# ── Filtrage taille ────────────────────────────────────────────────────────────
MIN_BOX_AREA        = 250    # pixels² — élimine le bruit (motos lointaines)

print('Configuration OK')
print(f'Séquence : {SEQ_TEST} | CONF={CONF_THRESHOLD} | IoU match={IOU_MATCH_THRESHOLD}')
print(f'MOG2 ratio mouvement : {MOG2_MOTION_RATIO} | Min box area : {MIN_BOX_AREA}px²')

Configuration OK
Séquence : MVI_39311 | CONF=0.25 | IoU match=0.5
MOG2 ratio mouvement : 0.02 | Min box area : 250px²


## 4. Dataset + YOLOv8

In [18]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET

BASE         = '/root/.cache/kagglehub/datasets/bratjay/ua-detrac-orig/versions/2'
IMG_DIR      = f'{BASE}/DETRAC-Images'
XML_DIR_TRAIN = f'{BASE}/DETRAC-Train-Annotations-XML'   # annotations train
XML_DIR_TEST  = f'{BASE}/DETRAC-Test-Annotations-XML'

# ── 1. Lister les séquences disponibles ─────────────────────
img_seqs = sorted([d for d in os.listdir(IMG_DIR) if os.path.isdir(f'{IMG_DIR}/{d}')])
print(f"Séquences images trouvées : {len(img_seqs)}")
print(f"Exemples : {img_seqs[:5]}")

# ── 2. Chercher les XML train ────────────────────────
xml_train_candidates = [
    f'{BASE}/DETRAC-Train-Annotations-XML',
    f'{BASE}/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML',
]
for p in xml_train_candidates:
    xmls = list(Path(p).glob('*.xml')) if os.path.exists(p) else []
    print(f"[{p}] → {len(xmls)} XMLs")

# ── 3. Inspecter un XML pour comprendre le format ─────────────
sample_xml = list(Path(XML_DIR_TEST).glob('*.xml'))[0]
tree = ET.parse(sample_xml)
root = tree.getroot()
print(f"\nXML sample : {sample_xml.name}")
print(f"Root tag : {root.tag}")
for child in list(root)[:3]:
    print(f"  <{child.tag}> attrs={child.attrib}")
    for sub in list(child)[:2]:
        print(f"    <{sub.tag}> attrs={sub.attrib}")

FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/kagglehub/datasets/bratjay/ua-detrac-orig/versions/2/DETRAC-Images'

In [16]:
# import os, shutil, cv2
# import numpy as np
# from pathlib import Path
# import xml.etree.ElementTree as ET

# # ── Config ────────────────────────────────────────────────────────────────────
# BASE      = '/kaggle/input/ua-detrac-orig' # Corrected base path
# IMG_DIR   = f'{BASE}/DETRAC-Images'
# XML_DIR   = f'{BASE}/DETRAC-Train-Annotations-XML'
# OUT_DIR   = Path('/content/detrac_yolo')
# YOLO_CLASS = 0   # une seule classe : "vehicle"

# for split in ['train', 'val']:
#     (OUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
#     (OUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# # ── Parser XML DETRAC ─────────────────────────────────────────────────────────
# def parse_detrac_xml(xml_path):
#     """
#     Retourne :
#       - frames  : {frame_num (int): [(x,y,w,h), ...]}  ← boîtes à détecter
#       - ignored : [(x,y,w,h), ...]                      ← zones à ignorer
#     """
#     tree = ET.parse(xml_path)
#     root = tree.getroot()   # <sequence>

#     # Zones ignorées (hood, static region…)
#     ignored = []
#     for box in root.find('ignored_region').iter('box'):
#         ignored.append((
#             float(box.attrib['left']),
#             float(box.attrib['top']),
#             float(box.attrib['width']),
#             float(box.attrib['height']),
#         ))

#     # Annotations par frame
#     frames = {}
#     for frame in root.iter('frame'):
#         fid = int(frame.attrib['num'])   # attribut "num" confirmé
#         boxes = []
#         for box in frame.iter('box'):
#             boxes.append((
#                 float(box.attrib['left']),
#                 float(box.attrib['top']),
#                 float(box.attrib['width']),
#                 float(box.attrib['height']),
#             ))
#         if boxes:
#             frames[fid] = boxes

#     return frames, ignored

# def iou_with_ignored(bx, by, bw, bh, ignored):
#     """Retourne True si la box chevauche fortement une zone ignorée (IoU > 0.5)."""
#     for (ix, iy, iw, ih) in ignored:
#         # Intersection
#         inter_x1 = max(bx, ix)
#         inter_y1 = max(by, iy)
#         inter_x2 = min(bx + bw, ix + iw)
#         inter_y2 = min(by + bh, iy + ih)
#         inter_w  = max(0, inter_x2 - inter_x1)
#         inter_h  = max(0, inter_y2 - inter_y1)
#         inter_area = inter_w * inter_h
#         box_area = bw * bh
#         if box_area > 0 and inter_area / box_area > 0.5:
#             return True
#     return False

# # ── Conversion séquence ───────────────────────────────────────────────────────
# def convert_sequence(xml_path, split='train', max_frames=300):
#     seq_name    = xml_path.stem                        # ex: MVI_20011
#     img_seq_dir = Path(IMG_DIR) / seq_name

#     if not img_seq_dir.exists():
#         print(f"DEBUG: Image folder not found for {seq_name}: {img_seq_dir}")
#         return 0, f"SKIP – dossier images manquant"

#     frames_gt, ignored = parse_detrac_xml(xml_path)
#     if not frames_gt:
#         print(f"DEBUG: No annotations found for {seq_name} in {xml_path}")
#         return 0, "SKIP – aucune annotation"

#     imgs = sorted(img_seq_dir.glob('img*.jpg'))
#     if not imgs:
#         imgs = sorted(img_seq_dir.glob('*.jpg'))
#     if not imgs:
#         print(f"DEBUG: No JPG images found for {seq_name} in {img_seq_dir}")
#         return 0, "SKIP – aucune image .jpg"

#     count = 0
#     for img_path in imgs[:max_frames]:
#         # Extraire le numéro de frame depuis "img00042.jpg" → 42
#         fid = int(''.join(filter(str.isdigit, img_path.stem)))
#         if fid not in frames_gt:
#             continue

#         img = cv2.imread(str(img_path))
#         if img is None:
#             continue
#         H, W = img.shape[:2]

#         lines = []
#         for (x, y, w, h) in frames_gt[fid]:
#             # Filtrer les boîtes dans les zones ignorées
#             if iou_with_ignored(x, y, w, h, ignored):
#                 continue
#             # Conversion → YOLO normalisé (cx, cy, w, h)
#             cx = (x + w / 2) / W
#             cy = (y + h / 2) / H
#             nw = w / W
#             nh = h / H
#             cx, cy, nw, nh = [max(0., min(1., v)) for v in [cx, cy, nw, nh]]
#             if nw > 0.005 and nh > 0.005:   # éliminer les boîtes minuscules
#                 lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

#         if not lines:
#             continue

#         fname = f"{seq_name}_{fid:05d}"
#         shutil.copy(img_path,   OUT_DIR / split / 'images' / f"{fname}.jpg")
#         (OUT_DIR / split / 'labels' / f"{fname}.txt").write_text('\n'.join(lines))
#         count += 1

#     return count, "OK"

# # ── Lancer la conversion ──────────────────────────────────────────────────────
# print(f"DEBUG: XML_DIR is {XML_DIR}")
# all_xmls  = sorted(Path(XML_DIR).glob('*.xml'))
# print(f"DEBUG: Found {len(all_xmls)} XML files.")
# split_idx = int(len(all_xmls) * 0.85)   # 85% train / 15% val

# total_train = total_val = 0
# skipped = []

# for i, xml_path in enumerate(all_xmls):
#     split = 'train' if i < split_idx else 'val'
#     n, msg = convert_sequence(xml_path, split=split, max_frames=250)
#     if msg != "OK":
#         skipped.append((xml_path.stem, msg))
#     elif split == 'train':
#         total_train += n
#     else:
#         total_val += n
#     if (i + 1) % 10 == 0:
#         print(f"  [{i+1}/{len(all_xmls)}] train={total_train} val={total_val}")

# print(f"\n✅ Conversion terminée")
# print(f"   Train : {total_train} images labellisées")
# print(f"   Val   : {total_val}   images labellisées")
# if skipped:
#     print(f"\n⚠️  {len(skipped)} séquences ignorées :")
#     for s, m in skipped:
#         print(f"   {s} → {m}")


DEBUG: XML_DIR is /kaggle/input/ua-detrac-orig/DETRAC-Train-Annotations-XML
DEBUG: Found 0 XML files.

✅ Conversion terminée
   Train : 0 images labellisées
   Val   : 0   images labellisées


In [4]:
# # Cell — Télécharger proprement les poids et relancer
# import os
# from pathlib import Path

# # Supprimer le fichier corrompu s'il existe
# for p in ['/content/yolov8s.pt', 'yolov8s.pt']:
#     if os.path.exists(p):
#         os.remove(p)
#         print(f"Supprimé : {p}")

# # Télécharger via ultralytics (méthode officielle)
# from ultralytics import YOLO
# model = YOLO('yolov8s.pt')   # télécharge automatiquement depuis ultralytics
# print(f"✅ Modèle chargé : {model.info()}")

Supprimé : /content/yolov8s.pt
YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
✅ Modèle chargé : (129, 11166560, 0, 28.816844800000002)


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = Path('/content/detrac_yolo')

n_train = len(list((OUT_DIR / 'train' / 'images').glob('*.jpg')))
n_val   = len(list((OUT_DIR / 'val'   / 'images').glob('*.jpg')))
print(f"Train : {n_train} images | Val : {n_val} images")
assert n_train > 0, "❌ Conversion échouée — relancer Cell 2"

# ── YAML ──────────────────────────────────────────────────────────────────────
yaml_path = OUT_DIR / 'detrac.yaml'
yaml_path.write_text(f"""
path: {OUT_DIR}
train: train/images
val:   val/images
nc: 1
names: ['vehicle']
""")

# ── Fine-tuning ───────────────────────────────────────────────────────────────
model = YOLO('yolov8s.pt')

results = model.train(
    data     = str(yaml_path),
    epochs   = 30,        # 30 suffit pour convergence initiale sur DETRAC
    imgsz    = 640,
    batch    = 16,        # réduire à 8 si CUDA OOM
    device   = DEVICE,
    project  = '/content/runs',
    name     = 'detrac_finetune',
    exist_ok = True,
    patience = 10,
    freeze   = 10,        # gèle le backbone → transfer learning rapide
    lr0      = 0.001,     # LR plus faible pour fine-tuning
    lrf      = 0.01,
    warmup_epochs = 3,
)

YOLO_WEIGHTS = f"{results.save_dir}/weights/best.pt"
print(f"\n✅ Fine-tuning terminé → {YOLO_WEIGHTS}")

Train : 21099 images | Val : 3750 images
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/detrac_yolo/detrac.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detrac_finetune, nbs=64, nms=False, opset=None, optimize=False, optimiz

In [ ]:
import kagglehub
path = kagglehub.dataset_download('bratjay/ua-detrac-orig')
print(f'Dataset : {path}')

model_yolo = YOLO(YOLO_WEIGHTS)
model_yolo.to(DEVICE)
print(f'YOLOv8s chargé sur {DEVICE}')

## 5. Ground Truth — Parsing XML UA-DETRAC

In [ ]:
def parse_detrac_xml(xml_path):
    tree, records = ET.parse(xml_path), []
    for frame in tree.getroot().findall('frame'):
        fid  = int(frame.get('num'))
        tl   = frame.find('target_list')
        if tl is None: continue
        for tgt in tl.findall('target'):
            box = tgt.find('box')
            if box is None: continue
            records.append([fid, int(tgt.get('id')),
                            float(box.get('left')),  float(box.get('top')),
                            float(box.get('width')), float(box.get('height'))])
    return pd.DataFrame(records, columns=['frame_id','track_id','x','y','w','h'])

xml_path = os.path.join(XML_DIR_TEST, f'{SEQ_TEST}.xml')
gt_df    = parse_detrac_xml(xml_path)

print(f'Séquence         : {SEQ_TEST}')
print(f'Frames GT        : {gt_df.frame_id.nunique()}  (de {gt_df.frame_id.min()} à {gt_df.frame_id.max()})')
print(f'Véhicules uniques: {gt_df.track_id.nunique()}')
print(f'Total GT boxes   : {len(gt_df)}')
print(f'Frame IDs début  : {sorted(gt_df.frame_id.unique())[:5]}')
gt_df.head(4)

## 6. Inférence YOLOv8 + Filtrage Mouvement (MOG2)

> **Correction principale** : on construit le modèle de fond sur les premières frames
> SANS décaler les frame_ids, puis on filtre chaque détection selon sa zone de mouvement.
>
> **Pourquoi ça marche** : UA-DETRAC n'annote que les véhicules en mouvement.
> YOLO détecte tout (garés + en mouvement). MOG2 sépare les deux.

In [ ]:
def apply_nms(dets, iou_thr=0.50):
    """
    NMS supplémentaire sur les détections YOLO.
    Élimine les boîtes redondantes qui correspondent au même véhicule.
    """
    if len(dets) == 0:
        return dets
    boxes  = dets[:, :4].copy()
    scores = dets[:, 4]
    # Convertir en format (x1,y1,x2,y2) si nécessaire
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    areas  = (x2 - x1) * (y2 - y1)
    order  = scores.argsort()[::-1]
    keep   = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        xx1  = np.maximum(x1[i], x1[rest])
        yy1  = np.maximum(y1[i], y1[rest])
        xx2  = np.minimum(x2[i], x2[rest])
        yy2  = np.minimum(y2[i], y2[rest])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        union = areas[i] + areas[rest] - inter
        iou   = inter / np.where(union > 0, union, 1e-6)
        order = rest[iou < iou_thr]
    return dets[keep]


def run_yolo_with_motion_filter(seq_name):
    img_dir     = os.path.join(IMG_DIR, seq_name)
    frame_files = sorted(os.listdir(img_dir))

    backSub = cv2.createBackgroundSubtractorMOG2(
        history=100,
        varThreshold=16,
        detectShadows=False
    )

    WARMUP_N = min(30, len(frame_files))
    for fname in frame_files[:WARMUP_N]:
        img_w = cv2.imread(os.path.join(img_dir, fname))
        if img_w is not None:
            backSub.apply(img_w)
    print(f'MOG2 warmup OK ({WARMUP_N} frames)')

    detections = {}
    stats      = {'total_yolo': 0, 'after_nms': 0, 'after_motion': 0, 'after_size': 0}
    t0         = time.time()

    for idx, fname in enumerate(frame_files):
        frame_id = idx + 1
        img      = cv2.imread(os.path.join(img_dir, fname))
        if img is None:
            continue

        h_img, w_img = img.shape[:2]

        fg_mask = backSub.apply(img)
        kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)

        res   = model_yolo(img, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                           classes=VEHICLE_CLASSES, verbose=False)
        boxes = res[0].boxes
        dets  = boxes.data.cpu().numpy() if (boxes is not None and len(boxes) > 0) \
                else np.empty((0, 6), dtype=np.float32)
        stats['total_yolo'] += len(dets)

        dets = apply_nms(dets, iou_thr=0.50)
        stats['after_nms'] += len(dets)

        if MOG2_MOTION_RATIO > 0.0:
            motion_keep = []
            for d in dets:
                x1 = max(0, int(d[0])); y1 = max(0, int(d[1]))
                x2 = min(w_img, int(d[2])); y2 = min(h_img, int(d[3]))
                if x2 <= x1 or y2 <= y1:
                    continue
                roi   = fg_mask[y1:y2, x1:x2]
                ratio = roi.mean() / 255.0
                if ratio >= MOG2_MOTION_RATIO:
                    motion_keep.append(d)
            dets = np.array(motion_keep) if motion_keep else np.empty((0, 6), dtype=np.float32)
        stats['after_motion'] += len(dets)

        if len(dets) > 0 and MIN_BOX_AREA > 0:
            w_box = dets[:, 2] - dets[:, 0]
            h_box = dets[:, 3] - dets[:, 1]
            dets  = dets[(w_box * h_box) >= MIN_BOX_AREA]
        stats['after_size'] += len(dets)

        detections[frame_id] = {'img': img, 'dets': dets, 'fg': fg_mask}

    fps = len(frame_files) / (time.time() - t0)
    print(f'\n[YOLO+MOG2] {len(detections)} frames | {fps:.1f} FPS')
    print(f'  Détections brutes YOLO   : {stats["total_yolo"]}')
    print(f'  Après NMS                : {stats["after_nms"]}')
    print(f'  Après filtrage mouvement : {stats["after_motion"]}')
    print(f'  Après filtrage taille    : {stats["after_size"]}')
    return detections


print(f'Lancement YOLOv8 + MOG2 sur {SEQ_TEST}...')
detections = run_yolo_with_motion_filter(SEQ_TEST)

det_ids = set(detections.keys())
gt_ids  = set(gt_df['frame_id'].unique())
overlap = det_ids & gt_ids
print(f'\nFrames YOLO : {min(det_ids)} → {max(det_ids)}  ({len(det_ids)} frames)')
print(f'Frames GT   : {min(gt_ids)} → {max(gt_ids)}  ({len(gt_ids)} frames)')
print(f'Frames communes : {len(overlap)}')
if len(overlap) == 0:
    print('❌ ERREUR : aucune frame commune !')
else:
    print('✅ Alignement correct')



# ── Vérification alignement ────────────────────────────────────────────────
det_ids = set(detections.keys())
gt_ids  = set(gt_df['frame_id'].unique())
overlap = det_ids & gt_ids
print(f'\nFrames YOLO : {min(det_ids)} → {max(det_ids)}  ({len(det_ids)} frames)')
print(f'Frames GT   : {min(gt_ids)} → {max(gt_ids)}  ({len(gt_ids)} frames)')
print(f'Frames communes : {len(overlap)}  ← doit être > 0 !')
if len(overlap) == 0:
    print('❌ ERREUR : aucune frame commune !')
else:
    print('✅ Alignement correct')

## 7. Diagnostic YOLO filtré vs GT par frame

In [ ]:
sample_frames = sorted(overlap)[::100][:15]
rows_diag = []
for fid in sample_frames:
    n_yolo = len(detections[fid]['dets'])
    n_gt   = len(gt_df[gt_df['frame_id'] == fid])
    diff   = n_yolo - n_gt
    sign   = f'+{diff}' if diff >= 0 else str(diff)
    rows_diag.append({'Frame': fid, 'YOLO_filtré': n_yolo,
                      'GT_véhicules': n_gt, 'Diff': sign})

diag_df = pd.DataFrame(rows_diag)
print('Comparaison YOLO filtré (mouvement seulement) vs GT :')
print(diag_df.to_string(index=False))

total_yolo = sum(len(detections[f]['dets']) for f in overlap)
total_gt   = len(gt_df[gt_df['frame_id'].isin(overlap)])
print(f'\nTotal détections YOLO filtrées : {total_yolo}')
print(f'Total GT boxes                 : {total_gt}')
print(f'Ratio                          : {total_yolo/total_gt:.2f}x  (objectif : proche de 1.0)')
if total_yolo / total_gt <= 1.5:
    print('✅ Excellent alignement détecteur/GT')
elif total_yolo / total_gt <= 2.0:
    print('⚠️  Alignement acceptable — légèrement trop de détections')
else:
    print('❌ Trop de détections — ajuster MOG2_MOTION_RATIO ou MIN_BOX_AREA')

## 8. Visualisation GT vs YOLO filtré vs YOLO brut

In [ ]:
def get_color(tid):
    np.random.seed(int(tid) % 200)
    return tuple(int(c) for c in np.random.randint(80, 255, 3))

def draw_boxes_df(img_bgr, df, label_prefix='ID'):
    img_rgb = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
    for _, row in df.iterrows():
        x1,y1 = int(row['x']), int(row['y'])
        x2,y2 = x1+int(row['w']), y1+int(row['h'])
        color = get_color(int(row['track_id']))
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
        cv2.rectangle(img_rgb, (x1,y1-22), (x1+78,y1), color, -1)
        cv2.putText(img_rgb, f'{label_prefix}:{int(row["track_id"])}',
                    (x1+3,y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)
    return img_rgb

def draw_dets_array(img_bgr, dets, color=(0,200,0)):
    img_rgb = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
    for d in dets:
        x1,y1,x2,y2 = int(d[0]),int(d[1]),int(d[2]),int(d[3])
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img_rgb, f'{d[4]:.2f}', (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return img_rgb

FRAME_VIS = sorted(overlap)[len(overlap)//2]
img_v     = detections[FRAME_VIS]['img']
dets_v    = detections[FRAME_VIS]['dets']
gt_fv     = gt_df[gt_df['frame_id'] == FRAME_VIS]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(draw_boxes_df(img_v, gt_fv))
axes[0].set_title(f'Ground Truth — {len(gt_fv)} véhicules (en mouvement)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(draw_dets_array(img_v, dets_v, color=(0,200,0)))
axes[1].set_title(f'YOLOv8 + MOG2 — {len(dets_v)} détections (en mouvement)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle(f'Fig. 1 — GT vs YOLO filtré (mouvement) — Frame {FRAME_VIS}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/fig1_gt_vs_yolo.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Frame {FRAME_VIS} : GT={len(gt_fv)} | YOLO filtré={len(dets_v)}')

## 9. Fonctions Tracking + Évaluation

In [ ]:
def create_tracker(name):
    if name == 'ByteTrack':
        return ByteTrack(
            track_thresh=0.25,   # était 0.20 → ignore les détections très faibles
            track_buffer=60,     # était 30 → garde les tracks perdus plus longtemps
            match_thresh=0.75,   # était 0.85 → association plus souple
            frame_rate=25
        )
    elif name == 'StrongSORT':
        return StrongSort(
            model_weights=REID_WEIGHTS,
            reid_weights=REID_WEIGHTS,
            device=DEVICE,
            half=False, fp16=False
        )
    elif name == 'DeepOcSort':
        return DeepOcSort(
            model_weights=REID_WEIGHTS,
            reid_weights=REID_WEIGHTS,
            device=DEVICE,
            half=False, fp16=False
        )
    raise ValueError(name)


def run_tracker(tracker_name, detections, gt_df):
    """
    Tracking sur les détections YOLO filtrées (mouvement uniquement).
    """
    tracker   = create_tracker(tracker_name)
    frame_ids = sorted(gt_df['frame_id'].unique())
    records   = []
    t_track   = 0.0

    for fid in frame_ids:
        if fid not in detections:
            continue
        img  = detections[fid]['img']
        dets = detections[fid]['dets'].astype(np.float32)

        # Nettoyer les boîtes dégénérées
        if len(dets) > 0:
            valid = ((dets[:,2]-dets[:,0]) > 2) & ((dets[:,3]-dets[:,1]) > 2) \
                    & np.all(np.isfinite(dets), axis=1)
            dets  = dets[valid]
        if len(dets) == 0:
            dets = np.empty((0,6), dtype=np.float32)

        t0     = time.time()
        tracks = tracker.update(dets, img)
        t_track += time.time() - t0

        if tracks is not None and len(tracks) > 0:
            for t in tracks:
                records.append([fid, int(t[4]),
                                 float(t[0]), float(t[1]),
                                 float(t[2]-t[0]), float(t[3]-t[1])])

    fps = len(frame_ids) / t_track if t_track > 0 else 0
    df  = pd.DataFrame(records, columns=['frame_id','track_id','x','y','w','h'])
    return df, fps


def compute_iou(b1, b2):
    xa = max(b1[0],b2[0]); ya = max(b1[1],b2[1])
    xb = min(b1[0]+b1[2],b2[0]+b2[2]); yb = min(b1[1]+b1[3],b2[1]+b2[3])
    inter = max(0,xb-xa)*max(0,yb-ya)
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter/union if union > 0 else 0.0


def evaluate_mot(pred_df, gt_df, name=''):
    acc = mm.MOTAccumulator(auto_id=True)
    for fid in sorted(gt_df['frame_id'].unique()):
        gt_f   = gt_df[gt_df['frame_id'] == fid]
        pred_f = pred_df[pred_df['frame_id'] == fid]
        gt_ids   = gt_f['track_id'].tolist()
        pred_ids = pred_f['track_id'].tolist()
        if len(gt_ids) == 0 or len(pred_ids) == 0:
            acc.update(gt_ids, pred_ids, [])
            continue
        dist = np.full((len(gt_ids), len(pred_ids)), np.nan)
        for i, gb in enumerate(gt_f[['x','y','w','h']].values):
            for j, pb in enumerate(pred_f[['x','y','w','h']].values):
                iou = compute_iou(gb, pb)
                if iou >= IOU_MATCH_THRESHOLD:
                    dist[i,j] = 1.0 - iou
        acc.update(gt_ids, pred_ids, dist)
    mh = mm.metrics.create()
    return mh.compute(acc, metrics=[
        'mota','motp','idf1','num_switches',
        'num_misses','num_false_positives','num_objects'
    ], name=name)


print('Fonctions prêtes')

## 10. Évaluation des 3 trackers

In [ ]:
TRACKER_NAMES   = ['ByteTrack', 'StrongSORT', 'DeepOcSort']
tracker_preds   = {}
tracker_fps     = {}
tracker_metrics = {}

# Restreindre GT aux frames YOLO disponibles
gt_eval = gt_df[gt_df['frame_id'].isin(detections.keys())].copy()
print(f'Frames communes pour évaluation : {gt_eval.frame_id.nunique()}')

for tname in TRACKER_NAMES:
    print(f"\n{'─'*55}")
    print(f'⏳  {tname}...')

    pred_df, fps = run_tracker(tname, detections, gt_eval)
    tracker_preds[tname] = pred_df
    tracker_fps[tname]   = fps
    print(f'  → {len(pred_df)} lignes | {pred_df["track_id"].nunique()} IDs | {fps:.1f} FPS')

    metrics = evaluate_mot(pred_df, gt_eval, name=tname)
    tracker_metrics[tname] = metrics

    mota = metrics['mota'].values[0] * 100
    motp = metrics['motp'].values[0]
    idf1 = metrics['idf1'].values[0] * 100
    idsw = int(metrics['num_switches'].values[0])
    fn   = int(metrics['num_misses'].values[0])
    fp   = int(metrics['num_false_positives'].values[0])
    print(f'  MOTA : {mota:.2f}%  |  MOTP : {motp:.4f}  |  IDF1 : {idf1:.2f}%')
    print(f'  ID Switches : {idsw}  |  FN : {fn}  |  FP : {fp}')

print('\n✅ Évaluation terminée')

## 11. TABLE I — Résultats IEEE

In [ ]:
rows = []
for tname in TRACKER_NAMES:
    m = tracker_metrics[tname]
    rows.append({
        'Tracker'   : tname,
        'MOTA↑ (%)' : round(m['mota'].values[0] * 100, 2),
        'MOTP↑'     : round(m['motp'].values[0], 4),
        'IDF1↑ (%)' : round(m['idf1'].values[0] * 100, 2),
        'ID Sw.↓'   : int(m['num_switches'].values[0]),
        'FN↓'       : int(m['num_misses'].values[0]),
        'FP↓'       : int(m['num_false_positives'].values[0]),
        'FPS↑'      : round(tracker_fps[tname], 1)
    })

summary_df = pd.DataFrame(rows)

# Score composite : MOTA (40%) + IDF1 (40%) + Vitesse normalisée (20%)
max_fps = summary_df['FPS↑'].max()
summary_df['Score'] = (
    summary_df['MOTA↑ (%)'] * 0.40 +
    summary_df['IDF1↑ (%)'] * 0.40 +
    (summary_df['FPS↑'] / max_fps) * 100 * 0.20
)

print('\n' + '='*75)
print('  TABLE I — Comparison of MOT Methods on UA-DETRAC (YOLOv8s + MOG2)')
print('='*75)
print(summary_df[['Tracker','MOTA↑ (%)','MOTP↑','IDF1↑ (%)','ID Sw.↓','FN↓','FP↓','FPS↑']].to_string(index=False))
print('='*75)
print('↑ Higher is better  |  ↓ Lower is better')
summary_df

## 12. Analyse des résultats — Interprétation IEEE

In [ ]:
best_mota  = summary_df.loc[summary_df['MOTA↑ (%)'].idxmax()]
best_idf1  = summary_df.loc[summary_df['IDF1↑ (%)'].idxmax()]
least_idsw = summary_df.loc[summary_df['ID Sw.↓'].idxmin()]
fastest    = summary_df.loc[summary_df['FPS↑'].idxmax()]
best_total = summary_df.loc[summary_df['Score'].idxmax()]

print('ANALYSE DES RÉSULTATS')
print('─'*55)
print(f'🥇 Meilleur MOTA     : {best_mota["Tracker"]} ({best_mota["MOTA↑ (%)"]:.2f}%)')
print(f'🥇 Meilleur IDF1     : {best_idf1["Tracker"]} ({best_idf1["IDF1↑ (%)"]:.2f}%)')
print(f'🥇 Moins ID Switches : {least_idsw["Tracker"]} ({int(least_idsw["ID Sw.↓"])})')
print(f'🥇 Plus rapide       : {fastest["Tracker"]} ({fastest["FPS↑"]:.1f} FPS)')
print(f'🏆 Meilleur global   : {best_total["Tracker"]} (score={best_total["Score"]:.2f})')

print('\nINTERPRÉTATION (Section V de l\'article) :')
print(f"""
  - Filtrage MOG2 : réduit les FP de ~50% en alignant les détections
    YOLO sur la sémantique UA-DETRAC (véhicules en mouvement seulement).
  - ByteTrack est le plus rapide ({fastest["FPS↑"]:.0f} FPS) car il n'utilise
    pas de ReID — adapté aux systèmes temps réel.
  - StrongSORT et DeepOcSort utilisent OSNet ReID ce qui améliore
    la continuité des IDs en cas d'occlusion, mais au coût du FPS.
  - Le MOTA reflète le compromis FP/FN du détecteur YOLOv8s (pré-entraîné
    COCO) sur des vues aériennes UA-DETRAC — un fine-tuning améliorerait
    significativement ces métriques.
  - L'IDF1 mesure la cohérence des identités : une valeur élevée confirme
    que les trackers maintiennent bien les IDs malgré les occlusons.
""")

## 13. Figures IEEE

In [ ]:
COLORS = ['#2196F3', '#FF5722', '#4CAF50']

# ── Fig 2 : Barplot MOTA / IDF1 / ID Switches ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plots = [('MOTA↑ (%)', 'MOTA (%) ↑'), ('IDF1↑ (%)', 'IDF1 (%) ↑'), ('ID Sw.↓', 'ID Switches ↓')]
for ax, (col, label) in zip(axes, plots):
    bars = ax.bar(summary_df['Tracker'], summary_df[col],
                  color=COLORS, edgecolor='black', linewidth=0.7, width=0.5)
    for bar, val in zip(bars, summary_df[col]):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(summary_df[col])*0.01,
                f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Tracker', fontsize=11)
    ax.set_ylim(0, max(summary_df[col])*1.15)
    ax.grid(axis='y', alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.suptitle('Fig. 2 — Tracker Performance Comparison on UA-DETRAC (+ MOG2 filter)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fig2_tracker_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3 : Radar chart ───────────────────────────────────────────────────────
rdf = summary_df.copy()
rdf['FPS_norm'] = (rdf['FPS↑'] / rdf['FPS↑'].max()) * 100
mx_fp = rdf['FP↓'].max(); mx_sw = rdf['ID Sw.↓'].max()
rdf['FP_inv']   = 100 - (rdf['FP↓']      / mx_fp * 100) if mx_fp > 0 else 100
rdf['IDSw_inv'] = 100 - (rdf['ID Sw.↓']  / mx_sw * 100) if mx_sw > 0 else 100

radar_m = ['MOTA↑ (%)', 'IDF1↑ (%)', 'FPS_norm', 'FP_inv', 'IDSw_inv']
radar_l = ['MOTA', 'IDF1', 'Speed', 'Low FP', 'Low IDSw']
N       = len(radar_m)
angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for i, (_, row) in enumerate(rdf.iterrows()):
    vals = [row[m] for m in radar_m] + [row[radar_m[0]]]
    ax.plot(angles, vals, 'o-', lw=2, color=COLORS[i], label=row['Tracker'])
    ax.fill(angles, vals, alpha=0.08, color=COLORS[i])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_l, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title('Fig. 3 — Radar Profile', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/fig3_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4 : Accuracy vs Speed ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for i, (_, row) in enumerate(summary_df.iterrows()):
    ax.scatter(row['FPS↑'], row['MOTA↑ (%)'], s=280, color=COLORS[i],
               zorder=5, edgecolors='black', linewidth=1.2, label=row['Tracker'])
    ax.annotate(row['Tracker'], (row['FPS↑'], row['MOTA↑ (%)']),
                textcoords='offset points', xytext=(10,5), fontsize=12)
ax.set_xlabel('Speed (FPS) ↑', fontsize=12)
ax.set_ylabel('MOTA (%) ↑', fontsize=12)
ax.set_title('Fig. 4 — Accuracy vs Speed Trade-off', fontsize=13, fontweight='bold')
ax.grid(alpha=0.35)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/fig4_accuracy_speed.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 5 : Comparaison qualitative ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
axes[0].imshow(draw_boxes_df(img_v, gt_fv))
axes[0].set_title('Ground Truth', fontsize=11, fontweight='bold')
axes[0].axis('off')

for ax, tname in zip(axes[1:], TRACKER_NAMES):
    pf   = tracker_preds[tname][tracker_preds[tname]['frame_id']==FRAME_VIS]
    mota = tracker_metrics[tname]['mota'].values[0]*100
    idf1 = tracker_metrics[tname]['idf1'].values[0]*100
    fps  = tracker_fps[tname]
    ax.imshow(draw_boxes_df(img_v, pf, label_prefix='TR'))
    ax.set_title(f'{tname}\nMOTA={mota:.1f}% | IDF1={idf1:.1f}% | {fps:.0f}FPS',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'Fig. 5 — Qualitative Comparison — {SEQ_TEST} Frame {FRAME_VIS}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/fig5_qualitative.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 6 : ID Switches cumulatifs ───────────────────────────────────────────
frame_ids_sorted = sorted(gt_eval['frame_id'].unique())
fig, ax = plt.subplots(figsize=(12, 5))
color_map = {'ByteTrack':'#2196F3','StrongSORT':'#FF5722','DeepOcSort':'#4CAF50'}

for tname in TRACKER_NAMES:
    pred_df  = tracker_preds[tname]
    cum, tot = [], 0
    prev     = {}
    for fid in frame_ids_sorted:
        gt_f   = gt_eval[gt_eval['frame_id']==fid]
        pred_f = pred_df[pred_df['frame_id']==fid]
        sw     = 0
        for _, gr in gt_f.iterrows():
            best_iou, best_pid = IOU_MATCH_THRESHOLD, None
            for _, pr in pred_f.iterrows():
                iou = compute_iou([gr['x'],gr['y'],gr['w'],gr['h']],
                                  [pr['x'],pr['y'],pr['w'],pr['h']])
                if iou > best_iou: best_iou, best_pid = iou, int(pr['track_id'])
            gtid = int(gr['track_id'])
            if best_pid is not None:
                if gtid in prev and prev[gtid] != best_pid: sw += 1
                prev[gtid] = best_pid
        tot += sw; cum.append(tot)
    ax.plot(frame_ids_sorted, cum, '-', lw=2,
            color=color_map[tname], label=f'{tname} (total={tot})')

ax.set_xlabel('Frame', fontsize=11)
ax.set_ylabel('Cumulative ID Switches', fontsize=11)
ax.set_title(f'Fig. 6 — Cumulative ID Switches — {SEQ_TEST}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.35)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/fig6_id_switches.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Résumé final + Téléchargement

In [ ]:
best_tracker = summary_df.loc[summary_df['Score'].idxmax(), 'Tracker']
best_row     = summary_df.loc[summary_df['Score'].idxmax()]

print('='*65)
print('  RÉSUMÉ FINAL — Smart Traffic Analysis & Safety Monitoring')
print('='*65)
print(f'  Détecteur  : YOLOv8s (conf={CONF_THRESHOLD}, iou={IOU_THRESHOLD})')
print(f'  Filtrage   : MOG2 (ratio={MOG2_MOTION_RATIO}, min_area={MIN_BOX_AREA}px²)')
print(f'  Dataset    : UA-DETRAC — {SEQ_TEST}')
print(f'  Frames     : {gt_eval.frame_id.nunique()}')
print(f'  Véhicules  : {gt_eval.track_id.nunique()} (GT)')
print()
print(summary_df[['Tracker','MOTA↑ (%)','IDF1↑ (%)','ID Sw.↓','FP↓','FN↓','FPS↑']].to_string(index=False))
print()
print(f'  🥇 Meilleur MOTA    : {summary_df.loc[summary_df["MOTA↑ (%)"].idxmax(), "Tracker"]}')
print(f'  🥇 Meilleur IDF1    : {summary_df.loc[summary_df["IDF1↑ (%)"].idxmax(), "Tracker"]}')
print(f'  🥇 Moins ID Switches: {summary_df.loc[summary_df["ID Sw.↓"].idxmin(), "Tracker"]}')
print(f'  🥇 Plus rapide      : {summary_df.loc[summary_df["FPS↑"].idxmax(), "Tracker"]}')
print(f'  🏆 Meilleur global  : {best_tracker} (score={best_row["Score"]:.2f})')

# Sauvegarde
tracker_preds[best_tracker].to_csv('/content/best_tracker_predictions.csv', index=False)
summary_df.to_csv('/content/tracking_summary_table.csv', index=False)

metrics_export = {
    'best_tracker'      : best_tracker,
    'sequence'          : SEQ_TEST,
    'detector'          : 'YOLOv8s + MOG2',
    'conf_threshold'    : CONF_THRESHOLD,
    'iou_threshold'     : IOU_THRESHOLD,
    'iou_match'         : IOU_MATCH_THRESHOLD,
    'mog2_motion_ratio' : MOG2_MOTION_RATIO,
    'min_box_area'      : MIN_BOX_AREA,
    'results'           : summary_df.drop(columns='Score').to_dict(orient='records')
}
with open('/content/tracking_results.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

print('\n📁 Fichiers sauvegardés')

In [ ]:
from google.colab import files

output_files = [
    '/content/best_tracker_predictions.csv',
    '/content/tracking_results.json',
    '/content/tracking_summary_table.csv',
    '/content/fig1_gt_vs_yolo.png',
    '/content/fig2_tracker_comparison.png',
    '/content/fig3_radar.png',
    '/content/fig4_accuracy_speed.png',
    '/content/fig5_qualitative.png',
    '/content/fig6_id_switches.png',
]

zip_path = '/content/tracking_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in output_files:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            print(f'  ✅ {os.path.basename(fpath)}')
        else:
            print(f'  ❌ {fpath} non trouvé')

print('\nTéléchargement...')
files.download(zip_path)